In [1]:
import pandas as pd
import pickle
from itertools import product
from xgboost import XGBClassifier
from score_functions import get_probs, draw_precision_recall_curve, draw_confusion_matrix, print_scores
from model_pipeline import get_train_test_splits

In [2]:
df = pd.read_csv("../data/processed/train_ohe.csv")
X_train, X_test, y_train, y_test = get_train_test_splits(df)
print(X_train.shape, y_train.shape)


(1037340, 83) (1037340,)


In [3]:
def run_xgboost(model_df:pd.DataFrame, X_train, y_train, X_test, y_test):
    
    param_grid = {
    "eta": [0.001, 0.01, 0.05, 0.1, 0.2, 0.3],
    "max_depth": [3, 5, 7, 9, 10],
    "n_estimators": [100, 200, 500, 700, 1000],
    "gamma": [0, 0.1, 0.2, 0.3],
    "subsample": [0.5, 0.7, 0.9],
    "min_child_weight": [1, 3, 5, 7],
    "colsample_bytree": [0.5, 0.7, 0.9],
    "reg_alpha": [0, 0.01, 0.1, 0.3, 1],
    }
    keys = list(param_grid.keys())
    model_num = 0
    for values in product(*(param_grid[k] for k in keys)):
        params = dict(zip(keys, values))
        model_name = f"xgboost_{model_num}"
        model_num += 1
        model = XGBClassifier(
            learning_rate=params["eta"],
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_child_weight=params["min_child_weight"],
            gamma=params["gamma"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            reg_alpha=params["reg_alpha"],
            random_state=4,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        print(model_name)
        y_pred_prob = get_probs(model, X_test)
        roc_auc, pr_auc, best_recall = print_scores(y_test, y_pred_prob, target_precision=0.95)
        model_df.loc[len(model_df)] = [
            model_name,
            params["eta"],
            params["max_depth"],
            params["n_estimators"],
            params["gamma"],
            params["subsample"],
            params["min_child_weight"],
            params["colsample_bytree"],
            params["reg_alpha"],
            roc_auc,
            pr_auc,
            best_recall,
        ]
        # save
        pickle.dump(model, open("./models/"+model_name+".pkl", "wb"))
        print("--------------------------------")
    print("training finished!")

In [4]:
model_df = pd.DataFrame(columns=[
    "model_name",
    "eta",
    "max_depth",
    "n_estimators",
    "gamma",
    "subsample",
    "min_child_weight",
    "colsample_bytree",
    "reg_alpha",
    "roc_auc",
    "pr_auc",
    "best_recall",
])


In [ ]:
run_xgboost(model_df, X_train, y_train, X_test, y_test)

xgboost_0
roc_auc: 0.9747331969171535, precision recall score: 0.6660928393310659, best recall at 0.95 precision: 0.31912058627581613
--------------------------------
xgboost_1
roc_auc: 0.9747331917493128, precision recall score: 0.6660920092089067, best recall at 0.95 precision: 0.31912058627581613
--------------------------------
xgboost_2
roc_auc: 0.9747346038617817, precision recall score: 0.6660678880231752, best recall at 0.95 precision: 0.31912058627581613
--------------------------------
xgboost_3
roc_auc: 0.9748223473369084, precision recall score: 0.6659769036092654, best recall at 0.95 precision: 0.3284477015323118
--------------------------------
xgboost_4
roc_auc: 0.9748518104886463, precision recall score: 0.6657241688752008, best recall at 0.95 precision: 0.3284477015323118
--------------------------------
xgboost_5
roc_auc: 0.9760585684718506, precision recall score: 0.6747017122659982, best recall at 0.95 precision: 0.3844103930712858
--------------------------------
x